In [1]:
import os, sys
import torch
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
DATA_DIR: str = "/content/drive/MyDrive/high-risk-data"
sys.path.insert(0, DATA_DIR)

from eval_utils import LABEL_COLS, score_predictions

data: pd.DataFrame = pd.read_csv(f"{DATA_DIR}/sbdh_splits.csv")
print(f"GPU: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print(f"rows: {len(data):,} | splits: {data['split'].value_counts().to_dict()}")
print(sorted(os.listdir(DATA_DIR)))

Mounted at /content/drive
GPU: True | Tesla T4
rows: 6,861 | splits: {'train': 4803, 'dev': 1029, 'test': 1029}
['__pycache__', 'bert_best.pt', 'dev_pred_bert.csv', 'dev_pred_llm_fewshot.csv', 'dev_pred_llm_zeroshot.csv', 'dev_pred_llm_zeroshot_v2.csv', 'dev_scores_all_models.csv', 'eval_utils.py', 'sbdh_splits.csv', 'test_pred_bert.csv', 'test_scores_bert_vs_regex.csv']


In [2]:
!pip install -q bitsandbytes

import json, re, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_NAME: str = "Qwen/Qwen2.5-7B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, quantization_config=bnb, device_map="auto")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

SYSTEM: str = """You extract social and behavioral determinants of health from the Social History section of a clinical note.
Return ONLY a JSON object with these 8 integer fields:
- sdoh_community_present: 1 if any active social support (lives with family/partner, involved family, etc.), else 0
- sdoh_community_absent: 1 if any loss of social support (widowed, divorced, separated, death of partner, estranged), else 0
- sdoh_education: 1 if any mention of education (school, college, degree), else 0
- sdoh_economics: 1 employed, 2 unemployed or retired or on disability, 0 not mentioned
- sdoh_environment: 1 any indication of housing, 2 homeless, 0 not mentioned
- behavior_alcohol, behavior_tobacco, behavior_drug: 1 current use, 2 past use (quit), 3 never, 4 ambiguous/unsure, 0 not mentioned"""

def build_prompt(text: str) -> str:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"Social History:\n{text}\n\nJSON:"}]
    return llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse_json(raw: str) -> dict[str, int] | None:
    m = re.search(r"\{.*?\}", raw, flags=re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return {c: int(obj[c]) for c in LABEL_COLS}
    except (json.JSONDecodeError, KeyError, ValueError, TypeError):
        return None

def llm_predict_one(text: str) -> dict[str, int] | None:
    inputs = llm_tok(build_prompt(text), return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=150, do_sample=False)
    return parse_json(llm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

# timing test: 10 dev notes, prints timing and parse rate only, no note text
dev_df = data[data["split"] == "dev"].reset_index(drop=True)
t0 = time.time()
results = [llm_predict_one(t) for t in dev_df["SOCIAL_HISTORY_TEXT"].head(10)]
elapsed = time.time() - t0
print(f"{elapsed / 10:.1f} s/note | parsed: {sum(r is not None for r in results)}/10")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

GPU memory used: 5.9 GB
6.6 s/note | parsed: 10/10


In [3]:
llm_tok.padding_side = "left"  # required for batched generation with decoder-only models
if llm_tok.pad_token is None:
    llm_tok.pad_token = llm_tok.eos_token

def llm_predict_batch(texts: list[str]) -> list[dict[str, int] | None]:
    prompts: list[str] = [build_prompt(t) for t in texts]
    inputs = llm_tok(prompts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=150, do_sample=False, pad_token_id=llm_tok.pad_token_id)
    gen = out[:, inputs["input_ids"].shape[1]:]
    return [parse_json(s) for s in llm_tok.batch_decode(gen, skip_special_tokens=True)]

sample: list[str] = dev_df["SOCIAL_HISTORY_TEXT"].head(32).astype(str).tolist()
for bs in (8, 16):
    t0 = time.time()
    res: list[dict[str, int] | None] = []
    for i in range(0, len(sample), bs):
        res.extend(llm_predict_batch(sample[i:i + bs]))
    el = time.time() - t0
    same = sum(a == b for a, b in zip(res[:10], results))
    print(f"batch {bs:>2}: {el / len(sample):.2f} s/note | parsed {sum(r is not None for r in res)}/32 | "
          f"matches unbatched on first 10: {same}/10 | GPU peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

batch  8: 2.62 s/note | parsed 32/32 | matches unbatched on first 10: 9/10 | GPU peak 7.1 GB
batch 16: 1.85 s/note | parsed 32/32 | matches unbatched on first 10: 10/10 | GPU peak 8.3 GB


In [4]:
import numpy as np

MAX_INPUT_TOKENS: int = 512

def truncate_text(text: str) -> str:
    ids = llm_tok(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= MAX_INPUT_TOKENS else llm_tok.decode(ids[:MAX_INPUT_TOKENS])

def predict_safe(texts: list[str]) -> list[dict[str, int] | None]:
    try:
        return llm_predict_batch(texts)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(texts) == 1:
            return [None]  # counted as a parse failure
        mid = len(texts) // 2
        return predict_safe(texts[:mid]) + predict_safe(texts[mid:])

def run_llm(df: pd.DataFrame, out_path: str, bs: int = 16) -> pd.DataFrame:
    done: pd.DataFrame = pd.read_csv(out_path) if os.path.exists(out_path) else pd.DataFrame()
    done_ids: set[int] = set(done["ROW_ID"]) if len(done) else set()
    todo: pd.DataFrame = df[~df["ROW_ID"].isin(done_ids)].sort_values("sh_len").reset_index(drop=True)
    print(f"already done: {len(done_ids)} | to run: {len(todo)}")
    t0 = time.time()
    for i in range(0, len(todo), bs):
        chunk = todo.iloc[i:i + bs]
        preds = predict_safe([truncate_text(t) for t in chunk["SOCIAL_HISTORY_TEXT"].astype(str)])
        rows = [{"ROW_ID": rid, "parsed": p is not None, **(p or {c: 0 for c in LABEL_COLS})}
                for rid, p in zip(chunk["ROW_ID"], preds)]
        pd.DataFrame(rows).to_csv(out_path, mode="a", header=not os.path.exists(out_path), index=False)
        if (i // bs) % 10 == 0:
            n = i + len(chunk)
            print(f"  {n}/{len(todo)} | {(time.time() - t0) / n:.2f} s/note")
    return pd.read_csv(out_path)

torch.cuda.empty_cache()
ZS_DEV: str = f"{DATA_DIR}/dev_pred_llm_zeroshot.csv"
zs_dev: pd.DataFrame = run_llm(dev_df, ZS_DEV)

zs_dev = dev_df[["ROW_ID"] + LABEL_COLS].merge(zs_dev, on="ROW_ID", suffixes=("_true", ""))
print(f"\nrows: {len(zs_dev)} | parse failures: {int((~zs_dev['parsed']).sum())}")
scores_zs: pd.DataFrame = score_predictions(
    zs_dev[[f"{c}_true" for c in LABEL_COLS]].set_axis(LABEL_COLS, axis=1),
    zs_dev[LABEL_COLS], "llm_zeroshot")

REGEX_DEV: dict[str, float] = {"sdoh_community_present": 0.871, "sdoh_community_absent": 0.795, "sdoh_education": 0.830,
    "sdoh_economics": 0.834, "sdoh_environment": 0.765, "behavior_alcohol": 0.541, "behavior_tobacco": 0.607, "behavior_drug": 0.522}
BERT_DEV: dict[str, float] = {"sdoh_community_present": 0.956, "sdoh_community_absent": 0.919, "sdoh_education": 0.890,
    "sdoh_economics": 0.904, "sdoh_environment": 0.802, "behavior_alcohol": 0.787, "behavior_tobacco": 0.860, "behavior_drug": 0.726}
cmp = pd.DataFrame({"regex": REGEX_DEV, "bert": BERT_DEV, "llm_zs": scores_zs.set_index("label")["macro_f1"]})
cmp.loc["MEAN"] = cmp.mean()
print(cmp.round(3).to_string())

already done: 1029 | to run: 0

rows: 1029 | parse failures: 0
                        regex   bert  llm_zs
sdoh_community_present  0.871  0.956   0.860
sdoh_community_absent   0.795  0.919   0.783
sdoh_education          0.830  0.890   0.394
sdoh_economics          0.834  0.904   0.652
sdoh_environment        0.765  0.802   0.351
behavior_alcohol        0.541  0.787   0.433
behavior_tobacco        0.607  0.860   0.533
behavior_drug           0.522  0.726   0.459
MEAN                    0.721  0.855   0.558


In [ ]:
from sklearn.metrics import confusion_matrix

for c in ["sdoh_education", "sdoh_environment", "sdoh_economics", "behavior_alcohol"]:
    labels = sorted(set(zs_dev[f"{c}_true"]) | set(zs_dev[c]))
    print(f"\n{c} (rows = true, cols = predicted, classes {labels})")
    print(confusion_matrix(zs_dev[f"{c}_true"], zs_dev[c], labels=labels))


sdoh_education (rows = true, cols = predicted, classes [0, 1, 2])
[[841 157   1]
 [  1  29   0]
 [  0   0   0]]

sdoh_environment (rows = true, cols = predicted, classes [0, 1, 2])
[[359   4   0]
 [560  92   1]
 [  3   8   2]]

sdoh_economics (rows = true, cols = predicted, classes [0, 1, 2])
[[370 115 136]
 [  4  98  41]
 [ 19  12 234]]

behavior_alcohol (rows = true, cols = predicted, classes [0, 1, 2, 3, 4])
[[198  16  31  15   7]
 [  6 107 119  52   4]
 [  0   4  41  11   0]
 [134   0  14 223   3]
 [ 14   7   5  16   2]]


In [5]:
SYSTEM_V2: str = """You extract social and behavioral determinants of health from the Social History section of a clinical note.
Follow these annotation rules exactly. Return ONLY a JSON object with these 8 integer fields.

- sdoh_community_present (0/1): 1 if the note shows ANY active social support: lives with someone, married, has a partner, has children or family, family involved or nearby. Otherwise 0.
- sdoh_community_absent (0/1): 1 if the note shows loss of social support: widowed, divorced, separated, partner died, estranged. "Lives alone" alone is not enough. Otherwise 0. Both community fields can be 1.
- sdoh_education (0/1): 1 ONLY if the note explicitly describes the patient's schooling or degree (high school, college, graduated, degree, grade level). A job title or profession is NOT education. Otherwise 0.
- sdoh_economics (0/1/2): 1 if currently employed or working; 2 if unemployed, retired, or on disability; 0 if current employment status is not stated. Do not guess from age or past jobs alone.
- sdoh_environment (0/1/2): 2 if homeless or in a shelter; 1 if the note says ANYTHING about where or with whom the patient lives (lives with wife, lives alone, lives at home, nursing home, apartment); 0 only if living situation is never mentioned.
- behavior_alcohol, behavior_tobacco, behavior_drug (0-4), each judged separately:
  1 = current use of ANY amount, including social, occasional, or rare use;
  2 = past use that has stopped (quit, former, remote history);
  3 = never or denies use; "denies tobacco, alcohol, and drugs" means 3 for EACH substance listed;
  4 = unclear or ambiguous wording;
  0 = substance not mentioned at all."""

VALID: dict[str, set[int]] = {c: set(range(N)) for c, N in
    {"sdoh_community_present": 2, "sdoh_community_absent": 2, "sdoh_education": 2, "sdoh_economics": 3,
     "sdoh_environment": 3, "behavior_alcohol": 5, "behavior_tobacco": 5, "behavior_drug": 5}.items()}

def clip_invalid(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    out, n_bad = df.copy(), 0
    for c in LABEL_COLS:
        bad = ~out[c].isin(VALID[c])
        n_bad += int(bad.sum())
        out.loc[bad, c] = 0
    return out, n_bad

# v1 rescored with clipping, to see how much of the education drop was the phantom class
zs_v1_clipped, n_bad_v1 = clip_invalid(zs_dev[["ROW_ID"] + LABEL_COLS])
true_dev = zs_dev[[f"{c}_true" for c in LABEL_COLS]].set_axis(LABEL_COLS, axis=1)
v1c = score_predictions(true_dev, zs_v1_clipped[LABEL_COLS], "v1_clipped").set_index("label")["macro_f1"]
print(f"v1 invalid outputs clipped: {n_bad_v1} | v1 mean (clipped): {v1c.mean():.3f}")

# switch the prompt to v2 and rerun dev
SYSTEM = SYSTEM_V2
ZS2_DEV: str = f"{DATA_DIR}/dev_pred_llm_zeroshot_v2.csv"
zs2 = run_llm(dev_df, ZS2_DEV)
zs2 = dev_df[["ROW_ID"]].merge(zs2, on="ROW_ID")
zs2, n_bad_v2 = clip_invalid(zs2)
v2 = score_predictions(true_dev, zs2[LABEL_COLS], "v2").set_index("label")["macro_f1"]

cmp2 = pd.DataFrame({"regex": REGEX_DEV, "bert": BERT_DEV, "llm_v1": v1c, "llm_v2": v2})
cmp2.loc["MEAN"] = cmp2.mean()
print(f"\nv2 parse failures: {int((~zs2['parsed']).sum())} | v2 invalid clipped: {n_bad_v2}")
print(cmp2.round(3).to_string())

v1 invalid outputs clipped: 1 | v1 mean (clipped): 0.583
already done: 1029 | to run: 0

v2 parse failures: 0 | v2 invalid clipped: 0
                        regex   bert  llm_v1  llm_v2
sdoh_community_present  0.871  0.956   0.860   0.893
sdoh_community_absent   0.795  0.919   0.783   0.787
sdoh_education          0.830  0.890   0.591   0.786
sdoh_economics          0.834  0.904   0.652   0.650
sdoh_environment        0.765  0.802   0.351   0.515
behavior_alcohol        0.541  0.787   0.433   0.536
behavior_tobacco        0.607  0.860   0.533   0.583
behavior_drug           0.522  0.726   0.459   0.378
MEAN                    0.721  0.855   0.583   0.641


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

K_SHOT: int = 4
EX_MAX_TOKENS: int = 200
train_df = data[data["split"] == "train"].reset_index(drop=True)

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
train_vecs = tfidf.fit_transform(train_df["SOCIAL_HISTORY_TEXT"].astype(str))

def trunc(text: str, n: int) -> str:
    ids = llm_tok(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= n else llm_tok.decode(ids[:n])

def build_prompt_fs(text: str) -> str:
    sims = linear_kernel(tfidf.transform([text]), train_vecs).ravel()
    idx = sims.argsort()[::-1][:K_SHOT][::-1]  # most similar example placed last, closest to the query
    msgs = [{"role": "system", "content": SYSTEM_V2}]
    for j in idx:
        ex = train_df.iloc[j]
        msgs.append({"role": "user", "content": f"Social History:\n{trunc(str(ex['SOCIAL_HISTORY_TEXT']), EX_MAX_TOKENS)}\n\nJSON:"})
        msgs.append({"role": "assistant", "content": json.dumps({c: int(ex[c]) for c in LABEL_COLS})})
    msgs.append({"role": "user", "content": f"Social History:\n{trunc(text, MAX_INPUT_TOKENS)}\n\nJSON:"})
    return llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def predict_prompts(prompts: list[str]) -> list[dict[str, int] | None]:
    try:
        inputs = llm_tok(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = llm.generate(**inputs, max_new_tokens=150, do_sample=False, pad_token_id=llm_tok.pad_token_id)
        return [parse_json(s) for s in llm_tok.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(prompts) == 1:
            return [None]
        mid = len(prompts) // 2
        return predict_prompts(prompts[:mid]) + predict_prompts(prompts[mid:])

def run_llm_fs(df: pd.DataFrame, out_path: str, bs: int = 8) -> pd.DataFrame:
    done = pd.read_csv(out_path) if os.path.exists(out_path) else pd.DataFrame()
    done_ids = set(done["ROW_ID"]) if len(done) else set()
    todo = df[~df["ROW_ID"].isin(done_ids)].sort_values("sh_len").reset_index(drop=True)
    print(f"already done: {len(done_ids)} | to run: {len(todo)}")
    t0 = time.time()
    for i in range(0, len(todo), bs):
        chunk = todo.iloc[i:i + bs]
        preds = predict_prompts([build_prompt_fs(str(t)) for t in chunk["SOCIAL_HISTORY_TEXT"]])
        rows = [{"ROW_ID": rid, "parsed": p is not None, **(p or {c: 0 for c in LABEL_COLS})}
                for rid, p in zip(chunk["ROW_ID"], preds)]
        pd.DataFrame(rows).to_csv(out_path, mode="a", header=not os.path.exists(out_path), index=False)
        if (i // bs) % 20 == 0:
            n = i + len(chunk)
            print(f"  {n}/{len(todo)} | {(time.time() - t0) / n:.2f} s/note")
    return pd.read_csv(out_path)

torch.cuda.empty_cache()
FS_DEV: str = f"{DATA_DIR}/dev_pred_llm_fewshot.csv"
fs = run_llm_fs(dev_df, FS_DEV)
fs = dev_df[["ROW_ID"]].merge(fs, on="ROW_ID")
fs, n_bad_fs = clip_invalid(fs)
fsc = score_predictions(true_dev, fs[LABEL_COLS], "fewshot").set_index("label")["macro_f1"]

cmp3 = pd.DataFrame({"regex": REGEX_DEV, "bert": BERT_DEV, "llm_zs_v2": v2, "llm_fs": fsc})
cmp3.loc["MEAN"] = cmp3.mean()
print(f"\nfew-shot parse failures: {int((~fs['parsed']).sum())} | invalid clipped: {n_bad_fs}")
print(cmp3.round(3).to_string())

already done: 1029 | to run: 0

few-shot parse failures: 0 | invalid clipped: 0
                        regex   bert  llm_zs_v2  llm_fs
sdoh_community_present  0.871  0.956      0.893   0.865
sdoh_community_absent   0.795  0.919      0.787   0.799
sdoh_education          0.830  0.890      0.786   0.789
sdoh_economics          0.834  0.904      0.650   0.737
sdoh_environment        0.765  0.802      0.515   0.656
behavior_alcohol        0.541  0.787      0.536   0.632
behavior_tobacco        0.607  0.860      0.583   0.703
behavior_drug           0.522  0.726      0.378   0.553
MEAN                    0.721  0.855      0.641   0.717


In [ ]:
cmp_all = pd.DataFrame({"regex": REGEX_DEV, "bert": BERT_DEV, "llm_zs_v1": v1c, "llm_zs_v2": v2, "llm_fs": fsc})
cmp_all.loc["MEAN"] = cmp_all.mean()
cmp_all.round(4).to_csv(f"{DATA_DIR}/dev_scores_all_models.csv")
print("saved dev_scores_all_models.csv")

saved dev_scores_all_models.csv


In [7]:
test_df = data[data["split"] == "test"].reset_index(drop=True)
true_test = test_df[LABEL_COLS]

SYSTEM = SYSTEM_V2  # zero-shot uses build_prompt, which reads the global SYSTEM
zs_t = run_llm(test_df, f"{DATA_DIR}/test_pred_llm_zeroshot_v2.csv")
zs_t, bad_zs = clip_invalid(test_df[["ROW_ID"]].merge(zs_t, on="ROW_ID"))

fs_t = run_llm_fs(test_df, f"{DATA_DIR}/test_pred_llm_fewshot.csv")
fs_t, bad_fs = clip_invalid(test_df[["ROW_ID"]].merge(fs_t, on="ROW_ID"))

REGEX_TEST: dict[str, float] = {"sdoh_community_present": 0.884, "sdoh_community_absent": 0.785, "sdoh_education": 0.834,
    "sdoh_economics": 0.811, "sdoh_environment": 0.848, "behavior_alcohol": 0.580, "behavior_tobacco": 0.621, "behavior_drug": 0.529}
BERT_TEST: dict[str, float] = {"sdoh_community_present": 0.962, "sdoh_community_absent": 0.897, "sdoh_education": 0.913,
    "sdoh_economics": 0.897, "sdoh_environment": 0.868, "behavior_alcohol": 0.813, "behavior_tobacco": 0.848, "behavior_drug": 0.704}

final = pd.DataFrame({
    "regex": REGEX_TEST, "bert": BERT_TEST,
    "llm_zs_v2": score_predictions(true_test, zs_t[LABEL_COLS], "zs").set_index("label")["macro_f1"],
    "llm_fs": score_predictions(true_test, fs_t[LABEL_COLS], "fs").set_index("label")["macro_f1"],
})
final.loc["MEAN"] = final.mean()
final.round(4).to_csv(f"{DATA_DIR}/test_scores_all_models.csv")
print(f"parse failures: zs={int((~zs_t['parsed']).sum())}, fs={int((~fs_t['parsed']).sum())} | invalid: zs={bad_zs}, fs={bad_fs}")
print(final.round(3).to_string())

already done: 0 | to run: 1029
  16/1029 | 1.87 s/note
  176/1029 | 1.84 s/note
  336/1029 | 1.86 s/note
  496/1029 | 1.89 s/note
  656/1029 | 1.91 s/note
  816/1029 | 1.94 s/note
  976/1029 | 1.99 s/note
already done: 0 | to run: 1029
  8/1029 | 3.22 s/note
  168/1029 | 3.38 s/note
  328/1029 | 3.46 s/note
  488/1029 | 3.57 s/note
  648/1029 | 3.66 s/note
  808/1029 | 3.78 s/note
  968/1029 | 3.93 s/note
parse failures: zs=0, fs=1 | invalid: zs=0, fs=0
                        regex   bert  llm_zs_v2  llm_fs
sdoh_community_present  0.884  0.962      0.931   0.919
sdoh_community_absent   0.785  0.897      0.768   0.786
sdoh_education          0.834  0.913      0.850   0.827
sdoh_economics          0.811  0.897      0.613   0.727
sdoh_environment        0.848  0.868      0.554   0.705
behavior_alcohol        0.580  0.813      0.611   0.695
behavior_tobacco        0.621  0.848      0.599   0.730
behavior_drug           0.529  0.704      0.420   0.597
MEAN                    0.736  0.863  